# Halcyon generator — Colab

AudioGen (medium) + MusicGen (large) on a Colab GPU, exposed as a FastAPI endpoint via ngrok.

**Setup:** Runtime → Change runtime type → T4 GPU. Put your ngrok token in the config cell, run every cell, then paste the printed URL into the account page under **Generators → Google Colab**.

While `orcrobert/halcyon` is a private repository, also set `GITHUB_TOKEN` in the first cell to a GitHub token with `repo` read access — the install cannot clone the package without it. Leave it blank once the repository is public.

The session is yours: your runtime, your ngrok token, your models. The notebook installs the server from the repository but writes nothing back to it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/orcrobert/halcyon/blob/main/generators/colab/halcyon_generator.ipynb)

In [ ]:
GITHUB_TOKEN = ""

REPO = "github.com/orcrobert/halcyon"
REF = "main"
CREDS = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
CORE = f"halcyon-generator @ git+https://{CREDS}{REPO}@{REF}#subdirectory=generators/core"

!pip install av==12.0.0 -q
!pip install --no-deps git+https://github.com/facebookresearch/audiocraft.git -q
!pip install encodec lameenc num2words spacy demucs cloudpickle flashy torchmetrics torchdiffeq pyngrok xformers -q
!pip install "{CORE}" -q

In [ ]:
import os

NGROK_TOKEN = "YOUR_NGROK_TOKEN"

os.environ["AUDIOGEN_MODEL"] = "facebook/audiogen-medium"
os.environ["MUSICGEN_MODEL"] = "facebook/musicgen-large"
os.environ["GENERATION_DURATION"] = "10"
os.environ["GENERATION_DEVICE"] = "cuda"

In [ ]:
import threading

import torch
import uvicorn
from pyngrok import ngrok

from halcyon_generator import create_app

app = create_app()
print(f"Models ready. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000)
print(f"\nGENERATOR_URL={public_url}")

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000)).start()